# 02 - Avaliacao de Modelos

Comparacao detalhada dos 3 modelos treinados (with lags, no lags, advanced).

**Nota:** Os modelos devem ser treinados primeiro com `python scripts/retrain.py`.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
import json
import warnings

from src.features.feature_engineering import FeatureEngineer
from src.models.evaluation import ModelEvaluator

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

## 1. Carregar Metadados e Modelos

In [ ]:
models_dir = Path('../data/models')
evaluator = ModelEvaluator()

# Carregar metadados dos 3 variantes
variants = {}
variant_configs = {
    'with_lags': {
        'meta': 'metadata/training_metadata.json',
        'model': 'checkpoints/best_model.pkl',
        'features': 'features/feature_names.txt',
    },
    'no_lags': {
        'meta': 'metadata/training_metadata_no_lags.json',
        'model': 'checkpoints/best_model_no_lags.pkl',
        'features': 'features/feature_names_no_lags.txt',
    },
    'advanced': {
        'meta': 'metadata/metadata_advanced.json',
        'model': 'checkpoints/best_model_advanced.pkl',
        'features': 'features/advanced_feature_names.txt',
    },
}

for name, paths in variant_configs.items():
    meta_path = models_dir / paths['meta']
    model_path = models_dir / paths['model']
    feat_path = models_dir / paths['features']

    if not model_path.exists():
        print(f"  [{name}] Modelo nao encontrado: {model_path}")
        continue

    with open(meta_path) as f:
        meta = json.load(f)
    model = joblib.load(model_path)
    with open(feat_path) as f:
        feature_names = [l.strip() for l in f.readlines() if l.strip()]

    variants[name] = {'meta': meta, 'model': model, 'features': feature_names}
    print(f"  [{name}] {meta.get('best_model', 'N/A')} | {len(feature_names)} features | "
          f"RMSE={meta['test_metrics']['rmse']:.2f} | MAPE={meta['test_metrics']['mape']:.2f}%")

print(f"\n{len(variants)} variante(s) carregada(s)")

## 2. Reavaliar no Conjunto de Teste

In [ ]:
df = pd.read_parquet('../data/processed/processed_data.parquet')
fe = FeatureEngineer()

results = {}
for name, v in variants.items():
    if name == 'no_lags':
        df_feat = fe.create_features_no_lags(df)
    elif name == 'advanced':
        df_feat = fe.create_all_features(df, use_advanced=True)
    else:
        df_feat = fe.create_all_features(df)

    df_sorted = df_feat.sort_values('timestamp').reset_index(drop=True)
    test_start = int(0.85 * len(df_sorted))
    df_test = df_sorted.iloc[test_start:].copy()

    available = [f for f in v['features'] if f in df_test.columns]
    X_test = df_test[available].values
    y_test = df_test['consumption_mw'].values

    y_pred = v['model'].predict(X_test)
    metrics = evaluator.calculate_metrics(y_test, y_pred)
    results[name] = {'metrics': metrics, 'y_test': y_test, 'y_pred': y_pred, 'df_test': df_test}

    print(f"\n{name.upper()}:")
    for k, val in metrics.items():
        print(f"  {k}: {val:.4f}")

## 3. Comparacao Visual

In [ ]:
n_variants = len(results)
fig, axes = plt.subplots(1, n_variants, figsize=(7*n_variants, 5))
if n_variants == 1:
    axes = [axes]

for ax, (name, r) in zip(axes, results.items()):
    ax.scatter(r['y_test'][:2000], r['y_pred'][:2000], alpha=0.3, s=10)
    lims = [min(r['y_test'].min(), r['y_pred'].min()), max(r['y_test'].max(), r['y_pred'].max())]
    ax.plot(lims, lims, 'r--', linewidth=2)
    ax.set_xlabel('Real (MW)')
    ax.set_ylabel('Previsto (MW)')
    ax.set_title(f"{name}\nRMSE={r['metrics']['rmse']:.2f} | MAPE={r['metrics']['mape']:.2f}%")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Comparacao por Regiao

In [ ]:
for name, r in results.items():
    r['df_test'] = r['df_test'].copy()
    r['df_test']['abs_error'] = np.abs(r['y_test'] - r['y_pred'])

    regional = r['df_test'].groupby('region')['abs_error'].agg(['mean', 'std']).sort_values('mean')
    print(f"\n{name.upper()} - Erro por Regiao:")
    print(regional.to_string())

## 5. Tabela Comparativa

In [ ]:
comparison = pd.DataFrame({
    name: {k: round(v, 4) for k, v in r['metrics'].items()}
    for name, r in results.items()
}).T

print("=" * 70)
print("TABELA COMPARATIVA")
print("=" * 70)
print(comparison.to_string())

# Baseline comparison (from metadata)
print("\n\nCOMPARACAO COM BASELINES (do treino):")
for name, v in variants.items():
    bc = v['meta'].get('baseline_comparison', {})
    if bc:
        best_bl = min(bc.items(), key=lambda x: x[1].get('rmse', float('inf')))
        model_rmse = v['meta']['test_metrics']['rmse']
        bl_rmse = best_bl[1].get('rmse', 0)
        improvement = (1 - model_rmse / bl_rmse) * 100 if bl_rmse > 0 else 0
        print(f"  {name}: melhor baseline={best_bl[0]} (RMSE={bl_rmse:.2f}), "
              f"modelo RMSE={model_rmse:.2f}, melhoria={improvement:.1f}%")